In [41]:
import torch
import sys
from transformers import AutoTokenizer

In [9]:
model_name = "Qwen/Qwen2.5-0.5B"

tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [10]:
messages = [
    {"role": "user", "content": "What's 1+1?"},
    {"role": "assistant", "content": "The answer is 2"},
    {"role": "user", "content": "Why"},
    {"role": "assistant", "content": "You add 1 to 1, thus gets 2."},
]

In [11]:
formatted_str = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)

print(formatted_str)

<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
What's 1+1?<|im_end|>
<|im_start|>assistant
The answer is 2<|im_end|>
<|im_start|>user
Why<|im_end|>
<|im_start|>assistant
You add 1 to 1, thus gets 2.<|im_end|>
<|im_start|>assistant



In [21]:
system_msg = [{"role": "system", "content": ""}]

system_tokens = tokenizer.apply_chat_template(system_msg, tokenize=False, add_generation_prompt=False)
print(system_tokens)

<|im_start|>system
<|im_end|>



In [33]:
user_msg = [{"role": "system", "content": ""}, {"role": "user", "content": ""}]

user_tokens = tokenizer.apply_chat_template(user_msg, tokenize=False, add_generation_prompt=True)
print(f'---{user_tokens}---')

---<|im_start|>system
<|im_end|>
<|im_start|>user
<|im_end|>
<|im_start|>assistant
---


In [24]:
user_msg = [{"role": "system", "content": ""}, {"role": "user", "content": ""}, {"role": "assistant", "content": ""}]

user_tokens = tokenizer.apply_chat_template(user_msg, tokenize=False, add_generation_prompt=False)
print(user_tokens)

<|im_start|>system
<|im_end|>
<|im_start|>user
<|im_end|>
<|im_start|>assistant
<|im_end|>



In [25]:
init_prompt_size = 1
prompt_msg = messages[:init_prompt_size]
completion_msg = messages[init_prompt_size:]

prompt_msg = [{"role": "system", "content": ""}] + prompt_msg

prompt_tokens = tokenizer.apply_chat_template(prompt_msg, tokenize=False, add_generation_prompt=False)
print(prompt_tokens)

<|im_start|>system
<|im_end|>
<|im_start|>user
What's 1+1?<|im_end|>



In [26]:

completion_tokens = tokenizer.apply_chat_template(completion_msg, tokenize=False, add_generation_prompt=False)
print(completion_tokens)

<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>assistant
The answer is 2<|im_end|>
<|im_start|>user
Why<|im_end|>
<|im_start|>assistant
You add 1 to 1, thus gets 2.<|im_end|>



In [27]:
results = []
for msg in completion_msg:
    turn = tokenizer.apply_chat_template([msg], tokenize=False, add_generation_prompt=False)
    results.append(turn)
    print(turn)


<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>assistant
The answer is 2<|im_end|>

<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Why<|im_end|>

<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>assistant
You add 1 to 1, thus gets 2.<|im_end|>



In [ ]:

# Your example messages
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is 1+1"},
    {"role": "assistant", "content": "The answer is 2"},
    {"role": "user", "content": "Why"},
    {"role": "assistant", "content": "You add 1 to 1, thus gets 2."},
    # Example with tool call (content is None) - should not be masked
    # {"role": "user", "content": "How's the weather in Paris"},
    # {"role": "assistant", "content": None, "tool_calls": [{"id": "call_123", "function": {"name": "get_weather", "arguments": '{"location": "Paris"}'}}]},
    # {"role": "tool", "tool_call_id": "call_123", "name": "get_weather", "content": '{"temperature": 25, "unit": "celsius"}'},
    # {"role": "assistant", "content": "The weather in Paris is 25 degrees Celsius."},
]

init_prompt_size = 3

# --- Helper Function to Find Subsequence ---
def find_subsequence(main_list, sub_list):
    """Finds the start index of the first occurrence of sub_list within main_list."""
    if not sub_list:
        return -1 # Cannot find empty subsequence
    len_sub = len(sub_list)
    for i in range(len(main_list) - len_sub + 1):
        if main_list[i:i+len_sub] == sub_list:
            return i
    return -1

# --- Core Logic (Revised) ---

# 1. Tokenize the entire conversation *once*
#    Make sure add_generation_prompt=False for training data
input_ids = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=False
)

# 2. Initialize the loss mask with all zeros
loss_mask = [0] * len(input_ids)

# 3. Iterate through messages to identify assistant *content* tokens
current_pos = 0
for i, msg in enumerate(messages):
    # Tokenize the prefix *up to and including* the current message
    prefix_tokens = tokenizer.apply_chat_template(
        messages[:i+1],
        tokenize=True,
        add_generation_prompt=False
    )

    # The tokens added by *this* message (including template) are the suffix
    # of prefix_tokens compared to the previous position
    message_tokens = prefix_tokens[current_pos:]

    if msg['role'] == 'assistant' and i >= init_prompt_size:
        # Tokenize *only* the content of the assistant message
        # Handle cases where content might be None or empty (e.g., tool calls)
        content = msg.get('content')
        if content:
            # Important: Tokenize content *without* adding any special tokens
            # that the tokenizer might add by default (like BOS/EOS)
            content_tokens = tokenizer.encode(content, add_special_tokens=False)

            if content_tokens: # Proceed only if content is not empty after tokenization
                # Find the start index of the content tokens within the message_tokens
                content_start_in_message = find_subsequence(message_tokens, content_tokens)

                if content_start_in_message != -1:
                    # Calculate the global start and end indices in the full input_ids
                    global_content_start = current_pos + content_start_in_message
                    global_content_end = global_content_start + len(content_tokens)

                    # print(f"  Found content tokens at index {content_start_in_message} within message tokens.")
                    # print(f"  Global indices: {global_content_start} to {global_content_end}")

                    # Set the mask to 1 only for these content tokens
                    for j in range(global_content_start, global_content_end):
                        # Defensive check: ensure index is within bounds
                        if j < len(loss_mask):
                            loss_mask[j] = 1
                        else:
                             print(f"Warning: Index {j} out of bounds for loss_mask (length {len(loss_mask)}). Check tokenization consistency.", file=sys.stderr)
                             sys.stderr.flush()

                else:
                    # This indicates a potential problem with the chat template or tokenization
                    print(f"Warning: Could not find content tokens for assistant message {i} within its tokenized representation. Check template/content.", file=sys.stderr)
                    print(f"  Content: '{content}'", file=sys.stderr)
                    print(f"  Content Tokens: {content_tokens}", file=sys.stderr)
                    print(f"  Message Tokens: {message_tokens}", file=sys.stderr)
                    sys.stderr.flush()
                    # Depending on strictness, you might want to raise an error here
                    # raise ValueError(f"Could not locate content tokens for assistant message {i}")

    # Update the current position in the full input_ids sequence
    current_pos = len(prefix_tokens)

# --- Verification ---
if len(input_ids) != len(loss_mask):
    # This check should ideally not fail with the new logic, but good to keep.
    raise ValueError(
        f"Mismatch! Final input_ids length ({len(input_ids)}) "
        f"does not match loss_mask length ({len(loss_mask)})."
    )

# Convert to tensor or numpy array if needed for training
input_ids_tensor = torch.tensor(input_ids)
loss_mask_tensor = torch.tensor(loss_mask)

# --- Output and Verification ---
print("\n--- Results ---")
print(f"Final Input IDs Length: {len(input_ids)}")
print(f"Final Loss Mask Length: {len(loss_mask)}")

# Decode parts to visually check masking
print("\n--- Masking Check (Decoded Tokens) ---")
masked_tokens = 0
unmasked_tokens = 0
for token_id, mask_val in zip(input_ids, loss_mask):
    # Handle potential decoding errors for special tokens if needed
    try:
        token_str = tokenizer.decode([token_id])
    except Exception:
        token_str = f"[UNK:{token_id}]"
    print(f"Token: {token_str:<15} | ID: {token_id:<6} | Mask: {mask_val}")
    if mask_val == 1:
        unmasked_tokens += 1
    else:
        masked_tokens += 1
    sys.stdout.flush() # Ensure output is printed promptly

print(f"\nTotal Tokens: {len(input_ids_tensor)}")
print(f"Masked Tokens (Loss=0): {masked_tokens}")
print(f"Unmasked Tokens (Loss=1): {unmasked_tokens}")



--- Results ---
Final Input IDs Length: 56
Final Loss Mask Length: 56

--- Masking Check (Decoded Tokens) ---
Token: <|im_start|>    | ID: 151644 | Mask: 0
Token: system          | ID: 8948   | Mask: 0
Token: 
               | ID: 198    | Mask: 0
Token: You             | ID: 2610   | Mask: 0
Token:  are            | ID: 525    | Mask: 0
Token:  a              | ID: 264    | Mask: 0
Token:  helpful        | ID: 10950  | Mask: 0
Token:  assistant      | ID: 17847  | Mask: 0
Token: .               | ID: 13     | Mask: 0
Token: <|im_end|>      | ID: 151645 | Mask: 0
Token: 
               | ID: 198    | Mask: 0
Token: <|im_start|>    | ID: 151644 | Mask: 0
Token: user            | ID: 872    | Mask: 0
Token: 
               | ID: 198    | Mask: 0
Token: What            | ID: 3838   | Mask: 0
Token:  is             | ID: 374    | Mask: 0
Token:                 | ID: 220    | Mask: 0
Token: 1               | ID: 16     | Mask: 0
Token: +               | ID: 10     | Mask: 0
Token: 1       